# Applying QLoRA
## Introduction
Quantized Low-Rank Adaptation (QLoRA) is a cutting-edge fine-tuning technique designed to reduce memory and computational requirements while maintaining model performance drastically. It builds on Low-Rank Adaptation (LoRA) principles but adds quantization to the process, further reducing the size of the model’s weight matrices. This allows even large-scale language models to be fine-tuned on smaller hardware, making them accessible for more practical use cases.

In this reading, we’ll explore how QLoRA works, its advantages, and the steps to apply it effectively to fine-tune pretrained models.

By the end of this reading, you will be able to:
* Describe how QLoRA combines quantization and low-rank adaptation for efficient fine-tuning.
* Apply QLoRA to a pretrained model to reduce memory and computational costs.
* Fine-tune a quantized low-rank model on task-specific data and evaluate its performance.
* Optimize QLoRA for specific tasks by adjusting quantization levels and rank values.

## Why use QLoRA?
Traditional fine-tuning approaches require updating all the parameters in a model, which can be resource-intensive, especially for large models. LoRA addresses this issue by introducing low-rank adaptations, but even LoRA can require significant memory for very large models. QLoRA enhances the fine-tuning process by applying quantization, which reduces the precision of the model's weights (e.g., from 32-bit to 8-bit or even 4-bit), lowering the memory and computational requirements. Quantizing a model involves approximating the model's weight values to lower-precision numbers, significantly reducing the memory footprint while preserving much of the model's performance. This makes fine-tuning feasible on smaller hardware such as consumer graphics processing units (GPUs).

### Benefits of QLoRA
1. Lower memory requirements: by quantizing model parameters, QLoRA reduces the memory needed for storing and processing large models.
2. Reduced computational costs: similar to LoRA, QLoRA reduces the number of parameters that need to be fine-tuned. Quantization further reduces the computational burden.
3. Faster training: QLoRA allows for faster fine-tuning due to its smaller memory and computational requirements, making it ideal for rapid iterations.

## Step-by-step guide to fine-tune with QLoRA
The remaining of this reading will guide you through the following steps:
* Step 1: Data setup for QLoRA fine-tuning
* Step 2: Apply QLoRA to a pretrained model
* Step 3: Fine-tune the QLoRA-enhanced model
* Step 4: Evaluate the QLoRA-fine-tuned model
* Step 5: Optimize QLoRA for specific tasks

### Step 1: Data setup for QLoRA fine-tuning
To begin fine-tuning using QLoRA, you must set up your data properly. This includes preparing the dataset by splitting it into training, validation, and test sets. This step is crucial for ensuring that the model is trained effectively and can generalize well to unseen data.

#### Steps
1. Collect or load the dataset you want to use for fine-tuning.
2. Split the dataset into training (for model learning), validation (for tuning hyperparameters), and test sets (for evaluating performance).
3. Preprocess the data by tokenizing it, ensuring that it aligns with the input format expected by the model.

### Step 2: Apply QLoRA to a pretrained model
To apply QLoRA, you need to quantize the model and apply low-rank adaptations to specific layers, such as attention layers or feed-forward networks. QLoRA modifies these layers while keeping the rest of the model frozen.

In most cases, QLoRA allows you to choose which layers to quantize. You can experiment by quantizing only certain layers, such as the attention layers or feed-forward networks, rather than quantizing all layers. This flexibility allows you to explore different configurations and adjust the quantization to fit your specific task.

Both GPT-2 and BERT are pretrained transformer models widely used for natural language processing tasks. While GPT-2 is a generative model focusing on text generation, and BERT is optimized for tasks such as classification and question answering, they share a similar architecture based on the transformer model. This makes them both suitable candidates for QLoRA, demonstrating how the method can be applied to a variety of pretrained models.

#### Steps
1. Load a pretrained model (e.g., GPT-2, BERT).
2. Quantize the model to reduce precision.
3. Apply LoRA to specific layers.
4. Fine-tune the quantized low-rank matrices while freezing the rest of the parameters.

#### Explanation
In this example, the pretrained GPT-2 model is quantized to 8 bits, drastically reducing its memory requirements. LoRA is then applied to specific layers, such as attention heads, to ensure that only a small subset of parameters is fine-tuned.

In [3]:
#%pip install -q peft bitsandbytes accelerate
from transformers import AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Load GPT-2 in 8-bit quantized mode
bnb_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForSequenceClassification.from_pretrained(
    "gpt2",
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.pad_token_id = model.config.eos_token_id

# Prepare model for k-bit training, then apply LoRA
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["c_attn", "c_proj"],
)

quantized_model = get_peft_model(model, lora_config)
quantized_model.print_trainable_parameters()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 812,544 || all params: 125,253,888 || trainable%: 0.6487


### Step 3: Fine-tune the QLoRA-enhanced model
Once QLoRA is applied, the fine-tuning process begins. You will fine-tune the quantized model's low-rank matrices on your task-specific dataset, allowing the model to adapt to the task efficiently.

Steps
1. Prepare the dataset by splitting it into training, validation, and test sets.
2. Fine-tune the model using only the quantized low-rank matrices.

#### Explanation
The model is fine-tuned using the Trainer API, but only the quantized low-rank matrices are updated during training, making the process more efficient compared to traditional fine-tuning.

In [8]:
from transformers import Trainer, TrainingArguments, AutoTokenizer, DataCollatorWithPadding
from datasets import Dataset

# Build small demo datasets only if they are not already defined
if "train_data" not in globals() or "test_data" not in globals():
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    raw_train = Dataset.from_dict(
        {
            "text": [
                "This product is great and works perfectly.",
                "Terrible experience, I want a refund.",
                "Fast delivery and excellent quality.",
                "The item broke after one day."
            ],
            "label": [1, 0, 1, 0],
        }
    )

    raw_val = Dataset.from_dict(
        {
            "text": [
                "Very happy with this purchase.",
                "Not worth the money."
            ],
            "label": [1, 0],
        }
    )

    def preprocess(examples):
        tokens = tokenizer(examples["text"], truncation=True, max_length=128)
        tokens["labels"] = examples["label"]
        return tokens

    train_data = raw_train.map(preprocess, batched=True, remove_columns=["text", "label"])
    test_data = raw_val.map(preprocess, batched=True, remove_columns=["text", "label"])
else:
    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Set up training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
)

# Fine-tune the QLoRA-enhanced model
trainer = Trainer(
    model=quantized_model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=data_collator,
)

# Train the model
trainer.train()

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float

Epoch,Training Loss,Validation Loss
1,No log,1.170932
2,No log,1.168268
3,No log,1.105782


/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/peter.horstedt/git/AI-and-Machine-Learning/.venv/lib/python3.12/site-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


TrainOutput(global_step=3, training_loss=1.4192220369974773, metrics={'train_runtime': 2.4944, 'train_samples_per_second': 4.811, 'train_steps_per_second': 1.203, 'total_flos': 55643811840.0, 'train_loss': 1.4192220369974773, 'epoch': 3.0})

### Step 4: Evaluate the QLoRA-fine-tuned model
After fine-tuning, it’s important to evaluate the model’s performance on the test set to determine how well it generalizes to unseen data. While quantization can sometimes introduce small performance trade-offs, QLoRA aims to balance efficiency with high performance.

#### Explanation
After fine-tuning, the model is evaluated using the test set. Standard evaluation metrics such as accuracy, precision, recall, and F1 score can be used to assess the model’s performance.



In [11]:
# Evaluate the model on the test set (without triggering evaluate callback order issues)
pred_output = trainer.predict(test_data)

pred_labels = pred_output.predictions.argmax(axis=-1)
true_labels = pred_output.label_ids
test_accuracy = (pred_labels == true_labels).mean()

print(f"Test Accuracy: {test_accuracy:.4f}")

Test Accuracy: 0.5000


### Step 5: Optimize QLoRA for specific tasks
You can optimize QLoRA by adjusting the rank of the low-rank matrices or experimenting with different quantization levels. You can find the best balance between model efficiency and performance for your specific task by tuning these parameters.

#### Optimization ideas
* Adjust the rank of the low-rank matrices (e.g., increasing or decreasing the rank).
* Experiment with different quantization levels (e.g., 4-bit or 8-bit quantization) to see how they affect the model’s performance.
* Consider experimenting with other parameters, such as dropout rate, learning rate, or layer-wise adaptation, to see how they influence fine-tuning results. This provides additional flexibility in customizing the model for task-specific requirements.

In [12]:
import torch.nn as nn

def adjust_qlora_rank(peft_model, rank=4, adapter_name="default"):
	updated_layers = 0

	for module in peft_model.modules():
		if not (hasattr(module, "lora_A") and hasattr(module, "lora_B")):
			continue
		if adapter_name not in module.lora_A or adapter_name not in module.lora_B:
			continue

		old_a = module.lora_A[adapter_name]
		old_b = module.lora_B[adapter_name]

		new_a = nn.Linear(old_a.in_features, rank, bias=False).to(
			device=old_a.weight.device, dtype=old_a.weight.dtype
		)
		new_b = nn.Linear(rank, old_b.out_features, bias=False).to(
			device=old_b.weight.device, dtype=old_b.weight.dtype
		)

		module.lora_A[adapter_name] = new_a
		module.lora_B[adapter_name] = new_b

		if hasattr(module, "r") and adapter_name in module.r:
			module.r[adapter_name] = rank
		if hasattr(module, "lora_alpha") and hasattr(module, "scaling"):
			alpha = module.lora_alpha[adapter_name]
			module.scaling[adapter_name] = alpha / rank

		updated_layers += 1

	print(f"Updated LoRA rank to {rank} for {updated_layers} layers.")
	return peft_model

# Adjust the rank of the low-rank matrices
quantized_model = adjust_qlora_rank(quantized_model, rank=4)  # Experiment with different rank values

Updated LoRA rank to 4 for 36 layers.


## Conclusion
QLoRA is an advanced fine-tuning technique that combines the benefits of quantization and low-rank adaptation. By reducing the memory and computational requirements, QLoRA makes it feasible to fine-tune large models even on consumer-grade hardware. With careful application, QLoRA can deliver efficient fine-tuning without sacrificing performance, making it ideal for resource-constrained environments.